# Week 63, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w063d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class D:
    def __get__(self, obj, objtype=None):
        return "D"

class P:
    @property
    def x(self):
        return "property"

class A(P):
    y = D()

a = A()
a.__dict__.update(x="instance", y="instance")
print(a.x, a.y)

**Predict before you run:** what will this print?



In [ ]:
class Meta(type):
    def __call__(cls, *args):
        print("making", cls.__name__)
        return super().__call__(*args)

class Thing(metaclass=Meta):
    def __init__(self, n):
        print("init", n)

t = Thing(3)
print(type(Thing).__name__, type(t).__name__)

**Predict before you run:** what will this print?



In [ ]:
class Fallback:
    known = 1
    def __getattr__(self, name):
        return name * 2

f = Fallback()
print(f.known, f.ab, getattr(f, "zz", "default"))

**Predict before you run:** what will this print?



In [ ]:
def deco(cls):
    print("deco", cls.__name__)
    return cls

class Base:
    def __init_subclass__(cls, **kwargs):
        print("init_subclass", cls.__name__)

@deco
class Child(Base):
    print("body")

**Predict before you run:** what will this print?



In [ ]:
class Counter:
    total = 0
c = Counter()
c.total += 1
print(c.total, Counter.total)
del c.total
print(c.total)

### Exercise: Counting live instances

Write a class decorator `track_instances(cls)` that wraps the class's `__init__` so every new instance is added to a `weakref.WeakSet` (Week 62), and adds a class method `live_count()` returning how many live instances of **that** class (subclasses included) there are. Instances that are deleted stop counting at once.

In [ ]:
import weakref

def track_instances(cls):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_track():\n    \"a class decorator with weak references\"\n    @track_instances\n    class Window:\n        def __init__(self, title):\n            self.title = title\n    a, b = Window(\"a\"), Window(\"b\")\n    assert a.title == \"a\" and Window.live_count() == 2\n    del a\n    assert Window.live_count() == 1\n    class Dialog(Window):\n        pass\n    d = Dialog(\"d\")\n    assert Window.live_count() == 2 and Dialog.live_count() == 1\n    del b, d\n    assert Window.live_count() == 0")

### Exercise: Overriding attributes for a block

Write a context manager `overridden(obj, **values)` (Week 61) that sets the given attributes on `obj` for the duration of a `with` block and yields `obj`. Afterwards, even after an error, each attribute goes back to what the instance had before; if the instance had **no** value of its own (the name came from the class, or didn't exist), the instance attribute is deleted, so the class's value shows through again.

In [ ]:
from contextlib import contextmanager

@contextmanager
def overridden(obj, **values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_overridden():\n    \"a context manager over attributes\"\n    class Config:\n        mode = \"slow\"\n        def __init__(self):\n            self.level = 1\n    c = Config()\n    with overridden(c, level=5, mode=\"fast\") as same:\n        assert same is c and c.level == 5 and c.mode == \"fast\" and Config.mode == \"slow\"\n    assert c.level == 1 and c.mode == \"slow\" and \"mode\" not in vars(c)\n    try:\n        with overridden(c, level=9, extra=True):\n            raise KeyError\n    except KeyError:\n        pass\n    assert c.level == 1 and not hasattr(c, \"extra\")")

### Exercise: Forge: an object mapper

Write `Column`, `Model`, `unique` and `Database` to the specification above.

In [ ]:
REQUIRED = object()

class Column:
    def __init__(self, kind, *, default=REQUIRED, primary=False):
        ___

class Model:
    tables = {}
    ___

def unique(*names):
    ___

class Database:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_models():\n    \"columns and models\"\n    class Named(Model, abstract=True):\n        name = Column(str)\n    class Author(Named):\n        id = Column(int, primary=True)\n        born = Column(int, default=0)\n    assert Author._columns == (\"name\", \"id\", \"born\") and Model.tables[\"authors\"] is Author\n    a = Author(id=1, name=\"Chinua\")\n    assert a.born == 0 and repr(a) == \"Author(name='Chinua', id=1, born=0)\"\n    assert a.dirty() == [\"born\", \"id\", \"name\"] and isinstance(Author.name, Column)\n    for kwargs, err in (({\"id\": 2}, TypeError), ({\"id\": \"3\", \"name\": \"x\"}, TypeError), ({\"id\": 4, \"name\": \"x\", \"x\": 1}, TypeError)):\n        try:\n            Author(**kwargs)\n            assert False\n        except err:\n            pass\n    try:\n        a.id = 5\n        assert False\n    except AttributeError:\n        pass\n    a.id = 1\n    try:\n        class NoKey(Model):\n            title = Column(str)\n        assert False\n    except TypeError:\n        pass\n    class Book(Model, table=\"library\"):\n        isbn = Column(str, primary=True)\n    assert Model.tables[\"library\"] is Book and \"nameds\" not in Model.tables\n\ndef test_database():\n    \"saving, loading and finding\"\n    class Reader(Model):\n        id = Column(int, primary=True)\n        name = Column(str)\n        city = Column(str, default=\"Lagos\")\n    db = Database()\n    r1, r2 = Reader(id=2, name=\"Ada\"), Reader(id=1, name=\"Ben\", city=\"Accra\")\n    assert db.save(r1) == 2 and r1.dirty() == []\n    db.save(r2)\n    r1.city = \"Ibadan\"\n    assert r1.dirty() == [\"city\"]\n    assert db.get(Reader, 2) is r1 and db.readers.count() == 2\n    assert [r.name for r in db.readers.all()] == [\"Ben\", \"Ada\"]\n    db.save(r1)\n    db.forget()\n    fresh = db.get(Reader, 2)\n    assert fresh is not r1 and fresh.city == \"Ibadan\" and fresh.dirty() == [] and db.get(Reader, 2) is fresh\n    assert [r.id for r in db.readers.find_by_city(\"Accra\")] == [1] and db.readers.find_by_name(\"Zed\") == []\n    for bad in (lambda: db.readers.find_by_shoe, lambda: db.nothing, lambda: db.readers.other):\n        try:\n            bad()\n            assert False\n        except AttributeError:\n            pass\n    try:\n        db.get(Reader, 99)\n        assert False\n    except KeyError:\n        pass\n\ndef test_unique():\n    \"the class decorator\"\n    @unique(\"email\")\n    class Member(Model):\n        id = Column(int, primary=True)\n        email = Column(str)\n    db = Database()\n    db.save(Member(id=1, email=\"a@x.org\"))\n    db.save(Member(id=1, email=\"a@x.org\"))\n    try:\n        db.save(Member(id=2, email=\"a@x.org\"))\n        assert False\n    except ValueError:\n        pass\n    assert db.members.count() == 1\n    try:\n        @unique(\"phone\")\n        class Other(Model):\n            id = Column(int, primary=True)\n        assert False\n    except TypeError:\n        pass")

> **Four tools, four jobs.** Each tool in today's mapper does the job it's best at. Descriptors own one attribute each: type checks, the primary-key rule and change tracking live in one place and serve every model. __init_subclass__ runs for every model automatically, because registration and column collection must never be forgotten. The class decorator is opt-in and visible, right for a constraint only some models want. And __getattr__ handles a family of names (find_by_anything) that can't be listed in advance. No metaclass was needed: when the simpler tools fit, they keep the code readable and combinable with other libraries.

### Exercise: Forge bonus: observable models

User interfaces and caches often need to know when an object changes. Write `Watched(default=None)`, a data descriptor (values in the instance's `__dict__`, reading the default when unset) that reports every **real** change (new value not equal to the old one) to its instance, and `Observable`, a base class with:

- `watched`, a tuple of the class's `Watched` names (base classes first), collected by `__init_subclass__`;
- `subscribe(callback, *names)`: `callback(obj, name, old, new)` is called for changes to the named attributes (all watched attributes when no names are given; `AttributeError` for a name that isn't watched). It returns a function that unsubscribes (safe to call twice). Listeners belong to each instance;
- `batch()`: a context manager (reentrant, yielding the object) during which notifications are held back; when the **outermost** batch ends, even after an error, each attribute that changed gets **one** notification with its value before the batch and its final value, in the order the attributes first changed, and none if it ended up back where it started.

Set up per-instance state lazily (the first time it's needed), since subclasses' `__init__` methods might not call yours.

In [ ]:
from contextlib import contextmanager

class Watched:
    def __init__(self, default=None):
        ___

class Observable:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_notify():\n    \"change notifications\"\n    class Shape(Observable):\n        colour = Watched(\"black\")\n    class Box(Shape):\n        width = Watched(1)\n        label = \"plain\"\n    assert Box.watched == (\"colour\", \"width\")\n    b = Box()\n    seen, widths = [], []\n    stop = b.subscribe(lambda obj, n, old, new: seen.append((n, old, new)))\n    b.subscribe(lambda obj, n, old, new: widths.append(new), \"width\")\n    b.colour = \"red\"\n    b.colour = \"red\"\n    b.width = 5\n    b.label = \"x\"\n    assert seen == [(\"colour\", \"black\", \"red\"), (\"width\", 1, 5)] and widths == [5]\n    stop()\n    stop()\n    b.width = 6\n    assert len(seen) == 2 and widths == [5, 6]\n    try:\n        b.subscribe(print, \"label\")\n        assert False\n    except AttributeError:\n        pass\n    assert Box().colour == \"black\" and Box().width == 1\n\ndef test_batch():\n    \"batched notifications\"\n    class Point(Observable):\n        x = Watched(0)\n        y = Watched(0)\n    p = Point()\n    seen = []\n    p.subscribe(lambda obj, n, old, new: seen.append((n, old, new)))\n    with p.batch() as same:\n        assert same is p\n        p.y = 1\n        p.x = 5\n        p.y = 2\n        with p.batch():\n            p.x = 0\n        assert seen == []\n    assert seen == [(\"y\", 0, 2)]\n    try:\n        with p.batch():\n            p.x = 9\n            raise ValueError\n    except ValueError:\n        pass\n    assert seen[-1] == (\"x\", 0, 9) and p.x == 9\n    q = Point()\n    q.x = 3\n    assert len(seen) == 2")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [descriptors](w063d1.html), [how properties work](w063d2.html), [__getattr__](w063d3.html), [metaclasses](w063d4.html) or [class decorators](w063d5.html).

## Mixed review

## Forge: an object mapper

An **object mapper** lets you describe stored records as classes and work with them as objects. Write the four parts:

- **`Column(kind, *, default=REQUIRED, primary=False)`**, a data descriptor storing values in the instance's `__dict__`: setting a value that isn't a `kind` raises `TypeError`; changing a primary key that's already set to a **different** value raises `AttributeError`; and every set adds the column's name to the instance's set of changed columns, `obj.__dict__["_dirty"]`.
- **`Model`**, a base class using `__init_subclass__(cls, table=None, abstract=False)`: it collects every `Column` name (base classes first, in definition order) into `cls._columns`, a tuple. Unless the class is `abstract`, it must have **exactly one** primary column (`TypeError` otherwise), stored as `cls._primary`; its table name is `table` or the lower-cased class name plus `"s"`, stored as `cls._table`, and the class is registered in the dict `Model.tables`. `__init__(**values)` sets every column from the keywords or its default (`TypeError` for a missing required column or an unknown keyword). `dirty()` returns the sorted changed columns, `to_dict()` the column values and `repr` looks like `Reader(id=1, name='Ada', city='Lagos')`.
- **`unique(*names)`**, a class decorator storing the names in `cls._unique` (a `TypeError` for a name that isn't a column).
- **`Database()`**: `save(obj)` stores `obj.to_dict()` under its table and primary key, refusing (`ValueError`) a value in a unique column that another row (with a different key) already has; it clears the object's changed columns and returns the key. `get(cls, pk)` returns the **same object** for the same key while it stays remembered (an identity map), building a new instance from the stored row otherwise (with no changed columns), and raises `KeyError` for unknown keys; `forget()` clears the remembered objects. **`db.<table>`** (through `__getattr__`) returns a view of a registered table with `all()` (objects sorted by key), `count()` and `find_by_<column>(value)` (also through `__getattr__`), with `AttributeError` for unknown tables and columns.

## Forge bonus: observable models

## Weekly quiz

## Week 63 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can write data and non-data descriptors with __set_name__, storing values correctly. ([Day 1](w063d1.html))
- I can write property, staticmethod and classmethod myself and state the lookup order for reading and setting. ([Day 2](w063d2.html))
- I can use __getattr__, __getattribute__, __setattr__ and __delattr__ without recursion, and know what they don't see. ([Day 3](w063d3.html))
- I can explain class creation step by step and write metaclasses that check, register and control classes. ([Day 4](w063d4.html))
- I can write class decorators that add and wrap methods and generate code from annotations. ([Day 5](w063d5.html))
- I built an object mapper and an observable model. (Today)

Tomorrow is a **rest day**. Next week: **Inside CPython**: how Python runs your code, bytecode and `dis`, frames and the evaluation loop, objects in C, and the import system.